In [21]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from pydantic import BaseModel
from IPython.display import Image
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
import uuid

In [22]:
from dotenv import load_dotenv
load_dotenv("../LangChainModels/.env")

True

In [23]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]
    

In [24]:
llm =ChatOpenAI()

In [25]:
def chat_node(state: ChatState) -> ChatState:
    #take user query
    messages = state['messages']
    #send to llm
    response = llm.invoke(messages)
    #response store in state
    return{
        'messages': [response]
    }

In [26]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

#add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [27]:
thread_id = str(uuid.uuid4)

while True:
    user_input = input('type here: ')
    print('User: ', user_input)
    if user_input.strip().lower() in ['exit', 'quit', 'bye','end']:
        break
    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_input)]}, config=config)
    
    print('AI: ', response['messages'][-1].content)

User:  hi
AI:  Hello! How can I assist you today?
User:  my name is divya
AI:  Hello Divya! Nice to meet you. How can I help you today?
User:  what is my name 
AI:  Your name is Divya.
User:  what is 10+10
AI:  10 + 10 equals 20.
User:  add 30 to the result
AI:  Adding 30 to the result of 10 + 10 (which is 20) would give us 20 + 30 = 50.
User:  exit
